# 🎙️ AI Video Studio: RVC Cloud Model Trainer
This notebook is specifically designed to train your custom AI voice models (e.g., Mythia, Furina, or your own voice) using Google Colab's cloud GPUs for free, fast (~15 mins), and stable execution.

---

### 📁 Step 1: Connect to Your Google Drive (Highly Recommended)
Google Colab instances are temporary. By mounting your Google Drive, your voice datasets and trained voice models (`.pth` and `.index` files) will be saved permanently and will not be lost when the session closes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[SUCCESS] Google Drive mounted successfully!')

### ⚙️ Step 2: Restore Default Google Colab Environment (Quick & Clean)
This cell completely restores Google Colab's default system python path and configures a highly compatible pip package manager to ensure flawless installation of RVC packages.

In [ ]:
import os

print('[INFO] Auto-repairing any corrupted system python symlinks...')
# 1. Force remove any alternatives
os.system("sudo update-alternatives --remove-all python3 2>/dev/null || true")

# 2. Scan and find the actual working physical python binary in /usr/bin
for path in ['/usr/bin/python3.10', '/usr/bin/python3.11', '/usr/bin/python3.12', '/usr/bin/python3.9']:
    if os.path.exists(path):
        print(f"[INFO] Found physical python binary: {path}")
        # Recreate system symlink to physical file
        os.system(f"sudo ln -sf {path} /usr/bin/python3")
        print("[INFO] Recreated python3 symlink successfully!")
        break

print('[INFO] Installing compatible pip manager (pip<24.1) on system python...')
os.system("python3 -m pip install --upgrade 'pip<24.1'")

print('\n[SUCCESS] Google Colab system python environment restored successfully!')
os.system("python3 --version")

### 📦 Step 3: Clone RVC & Install Stable Dependencies (Ultra Fast - Under 1 Min)
This cell clones the RVC WebUI and installs the necessary machine learning libraries into the system environment. It uses the `--no-deps` flag for fairseq to bypass giant PyTorch/CUDA re-downloads (saving 3GB of traffic) and utilizes Colab's pre-installed GPU libraries instantly.

In [ ]:
# 1. Clean old directory if exists
%cd /content
!rm -rf /content/Retrieval-based-Voice-Conversion-WebUI

# 2. Clone the stable v1.0 RVC WebUI repository
print('[INFO] Cloning RVC WebUI repository...')
!git clone -b v1.0 https://github.com/camenduru/Retrieval-based-Voice-Conversion-WebUI.git

# 3. Enter directories
%cd /content/Retrieval-based-Voice-Conversion-WebUI

# 4. Lock in stable NumPy and SciPy first (Crucial to prevent NumPy 2.x conflicts)
print('[INFO] Installing stable legacy core (numpy 1.23.5, scipy 1.9.3, Cython)...')
!python3 -m pip install Cython numpy==1.23.5 scipy==1.9.3

# 5. Pre-install Fairseq lightweight support dependencies
print('[INFO] Pre-installing fairseq dependencies...')
!python3 -m pip install hydra-core==1.0.7 omegaconf==2.0.6 sacrebleu regex bitarray cffi

# 6. Install Fairseq from MiroPsota wheel with --no-deps (Installs instantly in 1 second!)
print('[INFO] Installing fairseq from precompiled wheel (no-deps)...')
!python3 -m pip install --no-deps "fairseq==0.12.2" --extra-index-url https://miropsota.github.io/torch_packages_builder

# 7. Install Numba & Librosa (Installs instantly from pre-compiled wheels!)
print('[INFO] Installing stable voice libraries (numba, librosa)...')
!python3 -m pip install numba==0.56.4 librosa==0.9.2

# 8. Install standard supporting libraries
print('[INFO] Installing pure-python support packages...')
!python3 -m pip install gradio gdown mega.py tensorboardX ffmpeg ffmpeg-python

# 9. Install Scientific and compiled packages
print('[INFO] Installing scientific suites (faiss-cpu, parselmouth, pyworld)...')
!python3 -m pip install faiss-cpu praat-parselmouth pyworld

# 10. Download pre-trained Hubert base model
print('[INFO] Downloading pre-trained hubert_base.pt...')
!wget https://huggingface.co/audo/VoiceConversionWebUI/resolve/main/hubert_base.pt -O hubert_base.pt

# 11. Create weights and logs folders
!mkdir -p /content/Retrieval-based-Voice-Conversion-WebUI/weights
!mkdir -p /content/Retrieval-based-Voice-Conversion-WebUI/logs

print('\n[SUCCESS] All RVC dependencies and base models installed successfully under system Python!')

### 🚀 Step 4: Launch RVC WebUI (Gradio Tunnel)
Run this cell to start the RVC WebUI server on Google Colab using system Python.

Once the startup log finishes, look for a blue link that says:
👉 **`Running on public URL: https://xxxxxxxx.gradio.live`**

**Click that public Gradio link** to open the RVC Studio interface in your browser and start training!

In [ ]:
%cd /content/Retrieval-based-Voice-Conversion-WebUI
print('[INFO] Starting RVC Server... Please wait for the public Gradio link to appear below.')
!python3 infer-web.py --colab --share